In [1]:
from gptnl_pii_mappers import (
    NER_FLAIR,
    PII_IBAN,
    NER_GLiNER,
    NER_XLM_RoBERTa,
    PII_CreditCardCVVCode,
    PII_CreditCardExpiryDate,
    PII_CreditCardNumber,
    PII_EuropeanVATNumber,
    PII_Flair_Large_Formatter,
    PII_GLiNERFormatter,
    PII_RobBERT_V2_Dutch_Formatter,
    PII_XLM_RoBERTa_Dutch_Formatter,
    chain_factory,
    ner_factory,
)


In [2]:
def details_of_formatter(formatter_class, test_str="Hello!"):
    """Prints the type of the formatter class and the type of an instance created from it.

    Args:
            formatter_class: The class from which an instance will be created.
            test_str (str): A test string. Defaults to "Hello!".

    Returns:
            None
    """
    print(type(formatter_class))
    print(type(formatter := formatter_class()))
    print(formatter.name, formatter.format(test_str))
    print(formatter.__class__.__name__)
    print(formatter_class.__name__)
    print(type(formatter := formatter_class()))

def read_file_content(file_path: str) -> str:
    """Reads the content of a file at the specified file path and returns it as a string.

    :param file_path: The path to the file that should be read.
    :return: The content of the file as a string.
    """
    from pathlib import Path

    path = Path(file_path)
    try:
        file_content = path.read_text(encoding='utf-8')
        return file_content
    except FileNotFoundError:
        print(f"Error: The file '{file_path}' was not found.")
        return ""
    except Exception as e:
        print(f"An error occurred: {e}")
        return ""

def write_to_file(file_path: str, content: str) -> None:
    """Writes the given content to a file at the specified file path.

    If the file does not exist, it will be created. If the file already exists,
    its contents will be overwritten.

    :param file_path: The path to the file where content should be written.
    :param content: The content to write to the file.
    """
    from pathlib import Path

    path = Path(file_path)
    try:
        path.write_text(content, encoding='utf-8')
    except Exception as e:
        print(f"An error occurred while writing to the file: {e}")

In [1]:
from pathlib import Path

from gptnl_pii_mappers import PII_PrivateAI_TNO

text = """Hello! Mijn naam is Jan Henk de 10de en dit is een test. We werken aan NER PII Detection. Gezamelijk met Mark Rutte en TNO bouwen we aan een origineel nieuw GPT model. Getrained op data waar geen persoonsgegevens in zitten. Jan Henk de 10de heeft een bedrijf genaamd Jan Henk B.V. waar hij elke dag bezig is met werken."""

# formatter = PII_PrivateAI_GDPR(api_endpoint="https://privateai-gpu.ada.tnods.nl/", replacement_type="MARKER", entity_grouping_window=1024)

# formatter = PII_PrivateAI_GDPR(api_endpoint="https://privateai-cpu.ada.tnods.nl/", replacement_type="SYNTHETIC", entity_grouping_window=1024, check_public_figure=True, record_entities=True)

# response, metadata = formatter.format(text, with_metadata=True)
# print(response)
# print(metadata)

pf_csv_files = [Path("./gptnl_pii_mappers/_backend/public_figures/names_aliases_en.csv")]

formatter = PII_PrivateAI_TNO(api_endpoint="https://privateai-cpu.ada.tnods.nl/", replacement_type="SYNTHETIC", entity_grouping_window=1024, check_public_figure=True, record_entities=True, public_figure_csv_files=pf_csv_files)

response, metadata = formatter.format(text, with_metadata=True)
print(response)
print(metadata)

Loading names from file_path: gptnl_pii_mappers/_backend/public_figures/names_aliases_en.csv
Hello! Mijn naam is Lea Hပ္t at 22nd en dit is een test. We werken aan NER PII Detection. Gezamelijk met Mark Rutte en TNO bouwen we aan een origineel nieuw GPT model. Getrained op data waar geen persoonsgegevens in zitten. Lea Hပ္t at 22nd heeft een bedrijf genaamd Jan 짐 B.V. waar hij elke dag bezig is met werken. 
{'entity_types': ['NAME', 'NAME_GIVEN'], 'entity_type_counts': [2, 1], 'not_removed_entities': ['Mark Rutte'], 'not_removed_entity_counts': [1], 'processed_entities': ['Lea Hပ္t at 22nd', 'Jan 짐'], 'processed_entity_counts': [2, 1], 'processed_entity_types': ['NAME', 'NAME_GIVEN']}


# Test Person and Organisation PIIs with wikidata validation

In [ ]:
from gptnl_pii_mappers import NER_RobBERT, ner_factory, is_not_on_wikidata, PII_XLM_RoBERTa_Dutch_Formatter

PII_Validated_NER = ner_factory(class_=PII_XLM_RoBERTa_Dutch_Formatter, validator=is_not_on_wikidata, **NER_RobBERT["person_and_organisation"], group=True)

ner_formatter = PII_Validated_NER()
text = """Hello! Mijn naam is Jan Henk de 10de en dit is een test. We werken aan NER PII Detection.
    Gezamelijk met Mark Rutte en TNO bouwen we aan een origineel nieuw GPT model.
    Getrained op data waar geen persoonsgegevens in zitten.
    Jan Henk de 10de heeft een bedrijf genaamd Jan Henk B.V. waar hij elke dag bezig is met werken."""
text, metadata = ner_formatter.format(text, language="nl", with_metadata=True)
print(text)
print(metadata)


In [ ]:
from gptnl_pii_mappers import PII_FilePath

details_of_formatter(PII_FilePath, test_str="To access the project files, please navigate to C:\\Users\\Evangeline Estes\\Documents\\ProjectFiles\\Report.docx")

In [ ]:
import pandas as pd

from gptnl_pii_mappers import PII_DutchVreemdelingenDocumentNumber

address_csv = pd.read_csv("data/gptnl_synthetic_pii_data/NL/PII_DutchVreemdelingenDocumentNumber.csv", delimiter=';')
formatter = PII_DutchVreemdelingenDocumentNumber()

for index, row in address_csv.iterrows():
    formatted_sentence = formatter.format(row["Sentence"], language="nl")
    print(index, formatted_sentence)


In [ ]:
from gptnl_pii_mappers import PII_Everything

pii_formatter = PII_Everything()
text_dutch = """
Helga Maria Jansen en Carla Li zaten samen in een klein theater in Rotterdam, waar een voorstelling van hun oude vriend, acteur Jan Vermeer, op het punt stond te beginnen. Helga M. Jansen, die altijd van theater had gehouden, was bijzonder enthousiast. Naast hen zaten nog wat oude bekenden uit hun studietijd: Mark Peters, een kunstcriticus met wie mevrouw M.J. Jansen vroeger college had gevolgd, en Linda Bakker, een fotografe die de avond vastlegde met haar camera. Mevrouw Li had echter haar twijfels over de voorstelling; ze was nooit een groot fan van Jan’s wat abstracte werk geweest. “We zullen zien, Carla,” zei Jansen glimlachend toen de lichten dimden, terwijl Li haar blik naar het podium richtte.
Na de voorstelling verzamelden de vrienden zich in de foyer, waar ze werden vergezeld door Emma Vos, de regisseur van de voorstelling, en Alex Mulder, een energieke jonge dramaturg. Li complimenteerde Emma op de meeslepende sfeer van de avond, hoewel ze toe moest geven dat sommige scènes haar niet hadden aangesproken. Jansen, altijd enthousiast, begon een gepassioneerde discussie met Mark en Alex over de artistieke keuzes die Jan had gemaakt. Terwijl Carla met Linda en Emma stond te praten, schoot Helga Maria haar een glimlach toe; ze was blij dat haar vriendin, ondanks haar twijfels, toch was meegekomen. Toen de avond ten einde liep, liepen mevrouw Li en Jansen samen de frisse nachtlucht in, pratend over alles wat ze hadden gezien en beleefd.
"""

print(text_dutch)
print(pii_formatter.format(text_dutch, language="nl"))

text_english = """
On a breezy autumn afternoon, Helga Maria Jansen and Carla Li sat in their favorite café, "The Wandering Bean," catching up after months apart. Helga M. Jansen, always punctual, had already ordered her tea by the time Carla, known as "Ms. Li" to her students, walked in. The café was bustling with familiar faces: Jacob Hill, a local artist sketching in the corner, and Rosa Fernandez, an old friend who ran a nearby bookstore, waved at them from across the room. Mevrouw M.J. Jansen, ever the planner, had saved a cozy spot by the window, where they could sip their drinks and watch the autumn leaves drift by.
As they settled into their conversation, other friends from their university days, like Martin Klein and Sarah Reilly, came in to grab a quick coffee, stopping briefly to chat. Helga couldn’t wait to tell Ms. Li about her new project at the architecture firm, but Carla’s eyes sparkled with excitement about her own plans to start a community garden. The two friends laughed and shared memories, as their conversations blended with the familiar sounds of coffee machines and quiet chatter around them. Finally, as the afternoon light began to fade, Li and Jansen hugged goodbye, promising they wouldn’t let so much time pass before their next meet-up.
"""

print(text_english)
print(pii_formatter.format(text_english, language="en"))

In [ ]:
text_dutch = """
Helga Maria Jansen en Carla Li zaten samen in een klein theater in Rotterdam, waar een voorstelling van hun oude vriend, acteur Jan Vermeer, op het punt stond te beginnen. Helga M. Jansen, die altijd van theater had gehouden, was bijzonder enthousiast. Naast hen zaten nog wat oude bekenden uit hun studietijd: Mark Peters, een kunstcriticus met wie mevrouw M.J. Jansen vroeger college had gevolgd, en Linda Bakker, een fotografe die de avond vastlegde met haar camera. Mevrouw Li had echter haar twijfels over de voorstelling; ze was nooit een groot fan van Jan’s wat abstracte werk geweest. “We zullen zien, Carla,” zei Jansen glimlachend toen de lichten dimden, terwijl Li haar blik naar het podium richtte.
Na de voorstelling verzamelden de vrienden zich in de foyer, waar ze werden vergezeld door Emma Vos, de regisseur van de voorstelling, en Alex Mulder, een energieke jonge dramaturg. Li complimenteerde Emma op de meeslepende sfeer van de avond, hoewel ze toe moest geven dat sommige scènes haar niet hadden aangesproken. Jansen, altijd enthousiast, begon een gepassioneerde discussie met Mark en Alex over de artistieke keuzes die Jan had gemaakt. Terwijl Carla met Linda en Emma stond te praten, schoot Helga Maria haar een glimlach toe; ze was blij dat haar vriendin, ondanks haar twijfels, toch was meegekomen. Toen de avond ten einde liep, liepen mevrouw Li en Jansen samen de frisse nachtlucht in, pratend over alles wat ze hadden gezien en beleefd.
"""

text_dutch = read_file_content("data/test_large_text.txt")
n_characters_to_print = 5000


In [ ]:
if len(text_dutch) < n_characters_to_print:
    print(text_dutch)

PII_Person_Flair = ner_factory(class_=PII_Flair_Large_Formatter, **NER_FLAIR["person"], group=True)
pii_formatter = PII_Person_Flair()
formatted_text_flair = pii_formatter.format(text_dutch, language="nl")
if len(text_dutch) < n_characters_to_print:
    print(formatted_text_flair)
write_to_file("data/flair_output.txt", formatted_text_flair)

PII_Person_GLiNER = ner_factory(class_=PII_GLiNERFormatter, **NER_GLiNER["person"], group=True)
pii_formatter = PII_Person_GLiNER()
formatted_text_gliner = pii_formatter.format(text_dutch, language="nl")
if len(text_dutch) < n_characters_to_print:
    print(formatted_text_gliner)
write_to_file("data/gliner_output.txt", formatted_text_gliner)

PII_Person_RoBERT_V2 = ner_factory(class_=PII_RobBERT_V2_Dutch_Formatter, **NER_XLM_RoBERTa["person"], group=True)
pii_formatter = PII_Person_RoBERT_V2()
formatted_text_robert_v2 = pii_formatter.format(text_dutch, language="nl")
if len(text_dutch) < n_characters_to_print:
    print(formatted_text_robert_v2)
write_to_file("data/RoBERT_V2_output.txt", formatted_text_robert_v2)

PII_Person_RoBERTa = ner_factory(class_=PII_XLM_RoBERTa_Dutch_Formatter, **NER_XLM_RoBERTa["person"], group=True)
pii_formatter = PII_Person_RoBERTa()
formatted_text_roberta = pii_formatter.format(text_dutch, language="nl")
if len(text_dutch) < n_characters_to_print:
    print(formatted_text_roberta)
write_to_file("data/RoBERTa_output.txt", formatted_text_roberta)

In [ ]:
PII_Banking = chain_factory(
    ordered_list_of_formatter_classes=[
        PII_CreditCardNumber,
        PII_IBAN,
        PII_EuropeanVATNumber,
        PII_CreditCardCVVCode,
        PII_CreditCardExpiryDate,
    ]
)


banking_formatter = PII_Banking()
print(banking_formatter.format("""Hierbij zijn enkele voorbeelden van persoonlijke gegevens:
Het IBAN-nummer van de gemeentelijke rekening voor Rotterdam is NL12BNGH0285133675. 
U kunt het ook geschreven vinden als NL12 BNGH 0285 1336 75 met spaties ertussen. 
Mijn creditcardgegevens zijn als volgt: nummer 3576 8292 3893 9730, beveiligingscode (cvv) 202, en deze verloopt op 12/29.
Let op: deze gegevens zijn alleen voor testgebruik en niet authentiek. Alvast bedankt voor de medewerking.
"""))

PII_Banking_GLiNER = ner_factory(class_=PII_GLiNERFormatter, entity_types=["credit card number", "iban", "european vat number", "credit card cvv code", "cvc code", "cvv code", "credit card expiry date", "expiry date"], supported_languages=NER_GLiNER["person"]["supported_languages"])
gliner_banking_formatter = PII_Banking_GLiNER()
print(gliner_banking_formatter.format("""Hierbij zijn enkele voorbeelden van persoonlijke gegevens:
Het IBAN-nummer van de gemeentelijke rekening voor Rotterdam is NL12BNGH0285133675.
U kunt het ook geschreven vinden als NL12 BNGH 0285 1336 75 met spaties ertussen.
Mijn creditcardgegevens zijn als volgt: nummer 3576 8292 3893 9730, beveiligingscode (cvv) 202, en deze verloopt op 12/29.
Let op: deze gegevens zijn alleen voor testgebruik en niet authentiek. Alvast bedankt voor de medewerking.
""", language="nl", with_metadata=True))


In [ ]:
from gptnl_pii_mappers import NER_FLAIR, PII_Flair_Large_Formatter, ner_factory

PII_Grouping_Person = ner_factory(class_=PII_Flair_Large_Formatter, **NER_FLAIR["person"], group=True)

grouping_person_formatter = PII_Grouping_Person()

text = """
On a breezy autumn afternoon, Helga Maria Jansen and Carla Li sat in their favorite café, "The Wandering Bean," catching up after months apart. Helga M. Jansen, always punctual, had already ordered her tea by the time Carla, known as "Ms. Li" to her students, walked in. The café was bustling with familiar faces: Jacob Hill, a local artist sketching in the corner, and Rosa Fernandez, an old friend who ran a nearby bookstore, waved at them from across the room. Mevrouw M.J. Jansen, ever the planner, had saved a cozy spot by the window, where they could sip their drinks and watch the autumn leaves drift by.
As they settled into their conversation, other friends from their university days, like Martin Klein and Sarah Reilly, came in to grab a quick coffee, stopping briefly to chat. Helga couldn’t wait to tell Ms. Li about her new project at the architecture firm, but Carla’s eyes sparkled with excitement about her own plans to start a community garden. The two friends laughed and shared memories, as their conversations blended with the familiar sounds of coffee machines and quiet chatter around them. Finally, as the afternoon light began to fade, Li and Jansen hugged goodbye, promising they wouldn’t let so much time pass before their next meet-up.
"""

print(text)

print(grouping_person_formatter.format(text, language="en"))


In [ ]:
from gptnl_pii_mappers import PII_Banking

details_of_formatter(
    PII_Banking,
    test_str="""Hierbij zijn enkele voorbeelden van persoonlijke gegevens:
        Het IBAN-nummer van de gemeentelijke rekening voor Rotterdam is NL12BNGH0285133675. 
        U kunt het ook geschreven vinden als NL12 BNGH 0285 1336 75 met spaties ertussen. 
        Mijn creditcardgegevens zijn als volgt: nummer 3576 8292 3893 9730, beveiligingscode (cvv) 202, en deze verloopt op 12/29.
        Let op: deze gegevens zijn alleen voor testgebruik en niet authentiek. Alvast bedankt voor de medewerking.
    """,
)

# Test different NER Models

In [ ]:
from gptnl_pii_mappers import (
    PII_Person_Flair_Large,
    PII_Person_GLiNER,
    PII_Person_RobBERT,
    PII_Person_XLM_RoBERTa,
)

details_of_formatter(PII_Person_Flair_Large, test_str="Hello! Mijn naam is Jan Henk de 10de en dit is een test. We werken aan NER PII Detection.")

details_of_formatter(PII_Person_GLiNER, test_str="Hello! Mijn naam is Jan Henk de 10de en dit is een test. We werken aan NER PII Detection.")

details_of_formatter(PII_Person_RobBERT, test_str="Hello! Mijn naam is Jan Henk de 10de en dit is een test. We werken aan NER PII Detection.")

details_of_formatter(PII_Person_XLM_RoBERTa, test_str="Hello! Mijn naam is Jan Henk de 10de en dit is een test. We werken aan NER PII Detection.")


# Test Simple Regex and Validated Regex PIIs

In [ ]:
from gptnl_pii_mappers import PII_IBAN, PII_EmailAddress

details_of_formatter(PII_EmailAddress, test_str="Hello! Welkom bij PII Detection 101, we ontvangen al uw antwoorden graag op pii_detection@tno.nl. Alvast bedankt & tot ziens!")

details_of_formatter(PII_IBAN, test_str="Het IBAN-nummer van de gemeente Den Haag is NL12BNGH0285133675 maar ook NL12 BNGH 0285 1336 75, dat laatste heeft ook spaties er tussen.")


# Custom NER & REGEX

In [ ]:
from gptnl_pii_mappers import HuggingFaceNERFormatter


class PII_CUSTOM_Formatter(HuggingFaceNERFormatter):
    model_name = "HUGGINFACE-NER-PII"

    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.space_replacer = "▁"

    def format_ner(self, text: str, language: str) -> tuple[str, dict]:
        if self.model is None:
            from transformers import (
                AutoModelForTokenClassification,
                AutoTokenizer,
                pipeline,
            )

            self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
            self.model = AutoModelForTokenClassification.from_pretrained(self.model_name)
            self.pipeline = pipeline("ner", model=self.model, tokenizer=self.tokenizer, device=self.device)

        return super().format_ner(text, language)


custom_settings = {
    "entity_types": "PER",
    "mask": "<person>",
    "name": "PII_CUSTOM",
    "supported_languages": ["sv", "de", "is", "da", "no", "nl"],
}

formatter = ner_factory(class_=PII_CUSTOM_Formatter, **custom_settings, group=True)()

formatted_text = formatter.format("Private text")


In [ ]:
from gptnl_pii_mappers import regex_factory

danish_cpr_number_dict = {
        "pattern": r"\b\d{6}-\d{4}\b",
        "mask": "<danish_cpr_number>",
        "description": "This PII regex identifies a Danish CPR number consisting of 10 digits in the format DDMMYY-XXXX.",
        "source": "-",
        "name": "PII_DanishCPRNumber",
        "keywords": ["CPR", "personnummer"],
        "keyword_range": 300,
    }

formatter = regex_factory(**danish_cpr_number_dict)()

formatted_text = formatter.format(text="Kundens CPR-nummer er 120375-1234, hvilket skal behandles fortroligt.")
print(formatted_text)